03 - Big Data Processing with PySpark

This notebook uses PySpark to process the COVID-19 Romania county-level dataset.

The goal is to analyze COVID-19 data using a Big Data framework.

In [1]:
import os

java_home = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"

os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = os.path.join(java_home, "bin") + ";" + os.environ["PATH"]

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("Java exists:", os.path.exists(os.path.join(java_home, "bin", "java.exe")))


JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot
Java exists: True


STEP 1 - Import PySpark Libraries

The required PySpark libraries are imported for distributed data analysis.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, sum, avg, max, desc, col

print("PySpark libraries imported successfully.")


PySpark libraries imported successfully.


STEP 2 - Create Spark Session

A Spark session is created to run the analysis locally.

In [3]:
spark = (
    SparkSession.builder
    .appName("COVID-19 Romania Analytics")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark session created.")


Spark session created.


STEP 3 - Load the Processed Dataset

The cleaned county-level dataset is loaded from the processed folder.

In [4]:
file_path = "../data/processed/covid_county_clean.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

print("Dataset loaded.")
print("Rows:", df.count())
print("Columns:", len(df.columns))


Dataset loaded.
Rows: 4624
Columns: 8


STEP 4 - Dataset Schema

This step shows the column names and data types detected by Spark.

In [5]:
df.printSchema()

root
 |-- County: string (nullable = true)
 |-- Confirmed: integer (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Month_Name: string (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Risk_Level: string (nullable = true)



STEP 5 - Top Counties by Total Confirmed Cases

This analysis shows the counties with the highest total confirmed COVID-19 cases.

The maximum confirmed value is used because confirmed cases are cumulative.

In [6]:
top_counties = (
    df.groupBy("County")
    .agg(max("Confirmed").alias("Total_Confirmed"))
    .orderBy(desc("Total_Confirmed"))
)

top_counties.show(10, truncate=False)

+--------------+---------------+
|County        |Total_Confirmed|
+--------------+---------------+
|Mun. București|4618           |
|Suceava       |4290           |
|Brașov        |2424           |
|Argeș         |2176           |
|Galați        |1677           |
|Vrancea       |1400           |
|Dâmbovița     |1253           |
|Prahova       |1183           |
|Iași          |1149           |
|Botoșani      |1108           |
+--------------+---------------+
only showing top 10 rows


STEP 6 - Average Confirmed Cases by County

This analysis shows the average confirmed cases for each county across the available period.

In [7]:
from pyspark.sql.functions import round

avg_confirmed_by_county = (
    df.groupBy("County")
    .agg(round(avg("Confirmed"), 2).alias("Average_Confirmed"))
    .orderBy(desc("Average_Confirmed"))
)

avg_confirmed_by_county.show(10, truncate=False)

+--------------+-----------------+
|County        |Average_Confirmed|
+--------------+-----------------+
|Suceava       |3206.81          |
|Mun. București|2060.13          |
|Brașov        |893.61           |
|Neamț         |729.6            |
|Botoșani      |681.15           |
|Galați        |630.54           |
|Vrancea       |621.76           |
|Arad          |598.27           |
|Mureș         |572.53           |
|Hunedoara     |553.23           |
+--------------+-----------------+
only showing top 10 rows


STEP 7 - Monthly Confirmed Cases

This analysis shows confirmed cases grouped by year and month.


In [8]:
monthly_cases = (
    df.groupBy("Year", "Month")
    .agg(sum("Confirmed").alias("Monthly_Confirmed"))
    .orderBy("Year", "Month")
)

monthly_cases.show(20, truncate=False)


+----+-----+-----------------+
|Year|Month|Monthly_Confirmed|
+----+-----+-----------------+
|2020|4    |220283           |
|2020|5    |507655           |
|2020|6    |607826           |
|2020|7    |679550           |
+----+-----+-----------------+



STEP 8 - Risk Level Distribution on Latest Date

This analysis shows how many counties are classified as Low, Medium, or High risk on the latest available date.

In [9]:
latest_date = df.agg(max("Date").alias("Latest_Date")).collect()[0]["Latest_Date"]

latest_df = df.filter(col("Date") == latest_date)

risk_distribution = (
    latest_df.groupBy("Risk_Level")
    .agg(count("*").alias("County_Count"))
    .orderBy(desc("County_Count"))
)

print("Latest date:", latest_date)

risk_distribution.show(truncate=False)


Latest date: 2020-07-21
+----------+------------+
|Risk_Level|County_Count|
+----------+------------+
|Medium    |23          |
|Low       |16          |
|High      |4           |
+----------+------------+



STEP 9 - Highest Confirmed Values

This analysis shows the records with the highest confirmed case values.

In [10]:
highest_confirmed_values = (
    df.select("County", "Date", "Confirmed", "Risk_Level")
    .orderBy(desc("Confirmed"))
)

highest_confirmed_values.show(10, truncate=False)

+--------------+----------+---------+----------+
|County        |Date      |Confirmed|Risk_Level|
+--------------+----------+---------+----------+
|Mun. București|2020-07-21|4618     |High      |
|Mun. București|2020-07-20|4467     |High      |
|Mun. București|2020-07-19|4360     |High      |
|Suceava       |2020-07-21|4290     |High      |
|Suceava       |2020-07-20|4268     |High      |
|Mun. București|2020-07-18|4260     |High      |
|Suceava       |2020-07-19|4257     |High      |
|Suceava       |2020-07-18|4251     |High      |
|Suceava       |2020-07-17|4233     |High      |
|Suceava       |2020-07-16|4218     |High      |
+--------------+----------+---------+----------+
only showing top 10 rows


STEP 10 - PySpark Summary

PySpark was used to process the COVID-19 Romania county-level dataset.

The analysis includes:
1. Top counties by confirmed cases
2. Average confirmed cases by county
3. Monthly confirmed cases
4. Risk level distribution on the latest date
5. Highest confirmed values

In [11]:
print("PySpark analysis completed successfully.")

PySpark analysis completed successfully.


In [12]:
spark.stop()

print("Spark session stopped.")

Spark session stopped.
